# EDA - Dataset clínico de 210 pacientes

**Proyecto:** TFM - Técnicas avanzadas de IA para predicción y caracterización de complicaciones en biopsias pulmonares.

**Objetivo del notebook**

Este notebook realiza un análisis exploratorio completo del dataset clínico de 210 pacientes.
El análisis principal parte de un CSV ya categorizado. Además, se usa el CSV original para describir las categorías clínicas antes de su codificación.

**Objetivos concretos del EDA**

- validar integridad básica del dataset
- estudiar las etiquetas de complicación
- analizar la distribución de variables clínicas y patológicas
- identificar desbalanceos y rareza de etiquetas
- generar visualizaciones

In [ ]:
# Importaciones y librerias
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="pastel")
matplotlib.rcParams['figure.figsize'] = (8, 5)
matplotlib.rcParams['axes.titlesize'] = 13
matplotlib.rcParams['axes.labelsize'] = 11
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [ ]:
# Ruta y carga de datos
DATA_PATH = Path('/home/mcribilles/tfm/clinical_data/210pacientes/clinical_data_limpios.csv')
RAW_DATA_PATH = DATA_PATH.with_name('clinical_data.csv')
print(DATA_PATH, '->', DATA_PATH.exists())
print(RAW_DATA_PATH, '->', RAW_DATA_PATH.exists())

df = pd.read_csv(DATA_PATH)
df_raw = pd.read_csv(RAW_DATA_PATH)
print('Shape limpio:', df.shape)
print('Shape original:', df_raw.shape)

En el limpio tenemos 52 variables porque las tenemos ya categorizadas, por eso en el original hay menos columnas.

## 1. Vista general del dataset

Comenzamos comprobando dimensiones, tipos de variables, nulos e identificadores únicos para verificar que el dataset está correctamente cargado y preparado para el análisis.

In [ ]:
df.info()

In [ ]:
print('Número de pacientes:', len(df))
print('Número de columnas:', df.shape[1])
print('Nulos totales:', int(df.isna().sum().sum()))
print('IDs únicos:', df['patient_id'].nunique())
print('Duplicados de ID:', int(df['patient_id'].duplicated().sum()))

display(df.head())

In [ ]:
print('Valores nulos por columna:')
display(df.isnull().sum().sort_values(ascending=False))

In [ ]:
# Estadisticas descriptivas, aunque la mayoria de columnas son binarias
display(df.describe(include='all'))

## 2. Perfil oncológico de la cohorte

El tipo de cáncer no se conserva en el CSV limpio, por lo que este gráfico se calcula a partir de `clinical_data.csv`.

Los valores `X`, vacíos y nulos se muestran como `No especificado`.

In [ ]:
# preparacion y grafico de categorias del CSV original
MARCADORES_AUSENCIA = {'', 'x', 'nan', 'none'}

def tabla_frecuencias_original(serie, multietiqueta=False, incluir_no_especificado=False):
    valores = serie.fillna('').astype(str).str.strip()
    if multietiqueta:
        valores = valores.str.split(',').explode().str.strip()

    es_ausencia = valores.str.lower().isin(MARCADORES_AUSENCIA)
    if incluir_no_especificado:
        valores = valores.mask(es_ausencia, 'No especificado')
    else:
        valores = valores.loc[~es_ausencia]

    tabla = valores.value_counts().rename_axis('categoría').reset_index(name='n_pacientes')
    tabla['porcentaje'] = (100 * tabla['n_pacientes'] / len(df_raw)).round(1)
    return tabla

# agrupacion clinica de denominaciones equivalentes antes de cualquier grafico
def normalizar_tipo_cancer(serie):
    tipos = (
        serie.fillna('No especificado')
        .astype(str)
        .str.strip()
        .replace({'': 'No especificado', 'X': 'No especificado', 'x': 'No especificado'})
    )
    return tipos.replace({
        'Carcinoma epidermoide': 'Epidermoide',
        'Carcinoma de célula no pequeña': 'Cáncer de célula no pequeña',
        'Metástasis cáncer de endometrio': 'Metástasis',
        'Metástasis mama': 'Metástasis',
        'Tumor indiferenciado': 'No especificado',
    })

tipo_cancer_normalizado = normalizar_tipo_cancer(df_raw['Tipo de cáncer'])
cancer_original = tabla_frecuencias_original(
    tipo_cancer_normalizado, incluir_no_especificado=True
)
patologias_original = tabla_frecuencias_original(
    df_raw['Patología pulmonar'], multietiqueta=True
)
riesgos_original = tabla_frecuencias_original(
    df_raw['Factor de riesgo'], multietiqueta=True
)

def grafico_frecuencias(tabla, titulo, color):
    datos = tabla.sort_values('n_pacientes')
    alto = max(4, 0.42 * len(datos))
    fig, ax = plt.subplots(figsize=(10, alto))
    sns.barplot(data=datos, y='categoría', x='n_pacientes', color=color, ax=ax)
    ax.bar_label(
        ax.containers[0],
        labels=[f"{n} ({p:.1f}%)" for n, p in zip(datos['n_pacientes'], datos['porcentaje'])],
        padding=3,
    )
    ax.set_title(titulo)
    ax.set_xlabel('Número de pacientes')
    ax.set_ylabel('')
    ax.set_xlim(0, datos['n_pacientes'].max() * 1.18)
    plt.tight_layout()
    plt.show()

grafico_frecuencias(
    cancer_original, 'Tipos de cáncer', 'mediumpurple'
)

### Complicación según tipo de cáncer

Se utiliza el tipo de cáncer del CSV original y se comprueba que el orden de pacientes coincida con el CSV limpio. Las categorías con muy pocos pacientes se mantienen en la tabla de soporte, pero no se representan en el gráfico para evitar interpretar porcentajes basados en uno o dos casos.

In [ ]:
# comprobacion: se une por posicion solo si ambos CSV tienen el mismo orden de pacientes
if not df['patient_id'].astype(str).equals(df_raw['patient_id'].astype(str)):
    raise ValueError('Los IDs del CSV limpio y original no están alineados.')

df_cancer = df[['patient_id', 'Complicacion_binaria']].copy()
# Se reutiliza la misma agrupacion aplicada al grafico del perfil oncologico
df_cancer['Tipo_cancer_original'] = tipo_cancer_normalizado

resumen_cancer = (
    df_cancer.groupby('Tipo_cancer_original', as_index=False)
    .agg(
        n_pacientes=('patient_id', 'size'),
        n_complicacion=('Complicacion_binaria', 'sum'),
    )
)
resumen_cancer['n_sin_complicacion'] = (
    resumen_cancer['n_pacientes'] - resumen_cancer['n_complicacion']
)
resumen_cancer['tasa_complicacion_pct'] = (
    100 * resumen_cancer['n_complicacion'] / resumen_cancer['n_pacientes']
).round(1)
resumen_cancer = resumen_cancer.sort_values('n_pacientes', ascending=False)
display(resumen_cancer)

MIN_N_CANCER = 5
cancer_plot = resumen_cancer.query('n_pacientes >= @MIN_N_CANCER').sort_values('tasa_complicacion_pct')

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=cancer_plot, x='tasa_complicacion_pct', y='Tipo_cancer_original',
    color='mediumpurple', ax=ax
)
ax.bar_label(
    ax.containers[0],
    labels=[f"{tasa:.1f}% (n={n})" for tasa, n in zip(
        cancer_plot['tasa_complicacion_pct'], cancer_plot['n_pacientes']
    )],
    padding=3,
)
ax.set(
    title=f'Tasa de complicación por tipo de cáncer (n ≥ {MIN_N_CANCER})',
    xlabel='Pacientes con complicación (%)', ylabel=''
)
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

Adenocarcinoma es el grupo dominante (89 pacientes; 61,8 % con complicación), seguido de epidermoide (38; 42,1 %), el grupo sin tipo especificado (35; 48,6 %), linfoma (12; 16,7 %) y microcítico (8; 50,0 %). Las categorías de cáncer de célula no pequeña, metástasis y neuroendocrino tienen el mínimo de representación empleado para el gráfico (5 pacientes cada una), por lo que sus tasas deben leerse con cautela. Las diferencias aparentes no deben considerarse asociaciones estadísticamente confirmadas: salvo adenocarcinoma, epidermoide y el grupo no especificado, el soporte es reducido. Las categorías con menos de cinco pacientes se excluyen del gráfico, pero permanecen documentadas en la tabla.

## 3. Estructura de variables

El dataset contiene:

- identificador de paciente
- edad
- etiquetas objetivo de complicación
- variables binarias de factores de riesgo
- variables binarias de patología pulmonar
- sexo codificado
- complicación binaria global.

In [ ]:
print('Columnas del dataset:')
for i, col in enumerate(df.columns):
    print(f'{i:02d}. {col}')

In [ ]:
# separacion de bloques de variables
label_cols = ['Derrame_pleural', 'Hemorragia', 'Neumotórax', 'Sin_complicación', 'Complicacion_binaria']
base_cols = ['patient_id', 'Edad', 'Sexo_binaria']
aux_exclude = set(base_cols + label_cols)
feature_cols = [c for c in df.columns if c not in aux_exclude]

print('Variables base:', base_cols)
print('Variables objetivo:', label_cols)
print('Número de variables clínicas/patológicas binarias:', len(feature_cols))

Dependiendo de la clasificación que queramos hacer se utilizará una variable objetivo u otras.

## 4. Análisis de etiquetas objetivo

Aquí analizamos el soporte de las complicaciones y comprobamos si la formulación del problema es compatible con una visión multietiqueta.

Antes de empezar, vamos a hacer una breve comprobación de la coherencia entre etiquetas:  

In [ ]:
multi_target_cols = ['Derrame_pleural', 'Hemorragia', 'Neumotórax']

checks = {
    'Complicacion_binaria coincide con alguna complicación específica':
        (df['Complicacion_binaria'] == (df[multi_target_cols].sum(axis=1) > 0).astype(int)).all(),

    'Sin_complicación coincide con ausencia de complicaciones específicas':
        (df['Sin_complicación'] == (df[multi_target_cols].sum(axis=1) == 0).astype(int)).all(),

    'Complicacion_binaria y Sin_complicación son complementarias':
        (df['Complicacion_binaria'] == 1 - df['Sin_complicación']).all()
}

display(pd.Series(checks, name='resultado'))

In [ ]:
label_summary = pd.DataFrame({
    'etiqueta': label_cols,
    'n_positivos': [int(df[c].sum()) for c in label_cols],
    'proporcion': [df[c].mean() for c in label_cols]
})

sin_complicacion_especifica = df[multi_target_cols].sum(axis=1) == 0

label_summary_multietiqueta = pd.concat([
    label_summary[label_summary['etiqueta'].isin(multi_target_cols)].copy(),
    pd.DataFrame({
        'etiqueta': ['Sin complicación específica'],
        'n_positivos': [int(sin_complicacion_especifica.sum())],
        'proporcion': [sin_complicacion_especifica.mean()]
    })
], ignore_index=True)
label_summary_binaria = label_summary[label_summary['etiqueta'].isin(['Sin_complicación', 'Complicacion_binaria'])].copy()

display(label_summary)

Sin complicación y Complicación binaria son complementarias. Para evitar mezclar niveles de información, el soporte se visualiza en dos gráficos: uno para las complicaciones específicas multietiqueta, incluyendo el grupo sin ninguna complicación específica, y otro para la etiqueta resumen binaria. En el primer gráfico, las barras de complicaciones pueden solaparse si un paciente tiene más de una complicación etiquetada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(
    data=label_summary_multietiqueta,
    x='etiqueta',
    y='n_positivos',
    color='mediumpurple',
    ax=axes[0]
)
axes[0].set_title('Soporte de complicaciones específicas')
axes[0].set_xlabel('Complicación específica / ausencia')
axes[0].set_ylabel('Pacientes')
axes[0].tick_params(axis='x', rotation=20)

sns.barplot(
    data=label_summary_binaria,
    x='etiqueta',
    y='n_positivos',
    color='darkseagreen',
    ax=axes[1]
)
axes[1].set_title('Soporte de etiqueta binaria global')
axes[1].set_xlabel('Etiqueta binaria')
axes[1].set_ylabel('Pacientes')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Soporte de complicaciones especificas y ausencia de complicacion

fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(
    data=label_summary_multietiqueta,
    x='etiqueta',
    y='n_positivos',
    color='mediumpurple',
    ax=ax
)

ax.set_title('Soporte de complicaciones específicas')
ax.set_xlabel('Complicación específica / ausencia')
ax.set_ylabel('Pacientes')
ax.tick_params(axis='x', rotation=20)

# Añadir el numero de pacientes encima de cada barra
ax.bar_label(ax.containers[0], padding=3)

plt.tight_layout()

plt.savefig('soporte_etiquetas_complicacion.png', dpi=300, bbox_inches='tight')

plt.show()


Vemos como si hacemos una clasificación binaria, la etiqueta objetivo está balanceada. Sin embargo, si cogemos multietiqueta (se puede dar varias complicaciones a la vez), la complicación de derrame pleural solo tiene un paciente, por lo que sería despreciable. Además, la clase mayoritaria es la de sin complicación, por lo que hay desbalanceo para este tipo de clasificación.

In [ ]:
# Numero de tipos de complicacion por paciente
multi_target_cols = ['Derrame_pleural', 'Hemorragia', 'Neumotórax']
df['n_complicaciones_etiquetadas'] = df[multi_target_cols].sum(axis=1)

comp_per_patient = df['n_complicaciones_etiquetadas'].value_counts().sort_index()
print(comp_per_patient)

plt.figure(figsize=(6,4))
sns.barplot(x=comp_per_patient.index.astype(str), y=comp_per_patient.values, color='teal')
plt.title('Número de tipos de complicación por paciente')
plt.xlabel('Nº de complicaciones etiquetadas')
plt.ylabel('Pacientes')
plt.show()

Realmente hay muy pocos pacientes que tienen varias complicaciones a la vez.

In [ ]:
# Coocurrencia entre complicaciones
cooc = df[multi_target_cols].T.dot(df[multi_target_cols])
display(cooc)

plt.figure(figsize=(6,5))
sns.heatmap(cooc, annot=True, fmt='d', cmap='Purples')
plt.title('Matriz de coocurrencia de complicaciones')
plt.show()

Entre las complicaciones, la más usual es el neumotórax.

In [ ]:
# Combinaciones exactas de etiquetas de complicacion
combo_counts = (
    df[multi_target_cols]
    .astype(int)
    .astype(str)
    .agg('-'.join, axis=1)
    .value_counts()
    .rename_axis('combinacion')
    .reset_index(name='n_pacientes')
)

display(combo_counts)

Solo tenemos 9 pacientes con dos complicaciones a la vez. Derrame pleural solo hay un paciente. 

## 5. Variables demográficas básicas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.histplot(data=df, x='Edad', bins=20, kde=True, ax=axes[0], color='salmon')
axes[0].set_title('Distribución de edad')

sns.boxplot(data=df, x='Complicacion_binaria', y='Edad', ax=axes[1], palette='pastel')
axes[1].set_title('Edad según complicación binaria')
axes[1].set_xlabel('Complicación binaria')

plt.tight_layout()
plt.show()

In [ ]:
#distribucion de edad y comparacion por etiquetas principales
target_cols = ["Hemorragia", "Neumotórax", "Sin_complicación"]

# Pasar a formato largo: una fila por paciente y etiqueta presente
edad_long = df[["patient_id", "Edad"] + target_cols].melt(
    id_vars=["patient_id", "Edad"],
    value_vars=target_cols,
    var_name="Etiqueta",
    value_name="Presente"
)

edad_long = edad_long[edad_long["Presente"] == 1].copy()

edad_long["Etiqueta"] = edad_long["Etiqueta"].replace({
    "Sin_complicación": "Sin complicación"
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribucion global de edad
sns.histplot(
    data=df,
    x="Edad",
    bins=20,
    kde=True,
    ax=axes[0],
    color="salmon"
)
axes[0].set_title("Distribución de edad")
axes[0].set_xlabel("Edad")
axes[0].set_ylabel("Pacientes")

# Edad en pacientes positivos por etiqueta
sns.boxplot(
    data=edad_long,
    x="Etiqueta",
    y="Edad",
    ax=axes[1],
    palette="pastel"
)

axes[1].set_title("Edad según etiqueta de complicación")
axes[1].set_xlabel("Etiqueta")
axes[1].set_ylabel("Edad")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()

plt.savefig("edad_hist_boxplot_complicacion.png", dpi=300, bbox_inches="tight")

plt.show()

Sin normalizar la edad: 

In [ ]:
DATA_PATH = Path('/home/mcribilles/tfm/clinical_data/210pacientes/clinical_data.csv')
RAW_DATA_PATH = DATA_PATH.with_name('clinical_data.csv')
df_raw = pd.read_csv(RAW_DATA_PATH)

In [ ]:
# Edad sin normalizar a partir del CSV original
# /home/mcribilles/tfm/clinical_data/210pacientes/clinical_data.csv

# Comprobamos que ambos CSV contienen los pacientes en el mismo orden
if not df['patient_id'].astype(str).equals(df_raw['patient_id'].astype(str)):
    raise ValueError(
        'Los IDs del CSV limpio y del CSV original no están alineados.'
    )

# Etiquetas 
target_cols = [
    'Hemorragia',
    'Neumotórax',
    'Sin_complicación'
]

# Creamos un dataframe auxiliar con:
# - patient_id y etiquetas del CSV procesado
# - Edad original SIN normalizar del CSV original
df_edad = df[['patient_id'] + target_cols].copy()
df_edad['Edad'] = pd.to_numeric(df_raw['Edad'], errors='coerce')


# formato largo
edad_long = df_edad.melt(
    id_vars=['patient_id', 'Edad'],
    value_vars=target_cols,
    var_name='Etiqueta',
    value_name='Presente'
)

#nos quedamos solo con los pacientes positivos para cada etiqueta
edad_long = edad_long[edad_long['Presente'] == 1].copy()

# Nombre más legible
edad_long['Etiqueta'] = edad_long['Etiqueta'].replace({
    'Sin_complicación': 'Sin complicación'
})


#grafica
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribución global de edad SIN normalizar
sns.histplot(
    data=df_edad,
    x='Edad',
    bins=20,
    kde=True,
    ax=axes[0],
    color='salmon'
)

axes[0].set_title('Distribución de edad')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Pacientes')


# Edad segun etiqueta de complicacion
sns.boxplot(
    data=edad_long,
    x='Etiqueta',
    y='Edad',
    ax=axes[1],
    palette='pastel'
)

axes[1].set_title('Edad según etiqueta de complicación')
axes[1].set_xlabel('Etiqueta')
axes[1].set_ylabel('Edad (años)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()

plt.savefig(
    'edad_sin_normalizar_hist_boxplot_complicacion.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# CSV con la edad original SIN normalizar
RAW_DATA_PATH = '/home/mcribilles/tfm/clinical_data/210pacientes/clinical_data.csv'

df_raw = pd.read_csv(RAW_DATA_PATH)


# Nos quedamos con patient_id y Edad del CSV original
edad_original = df_raw[['patient_id', 'Edad']].copy()

edad_original['Edad'] = pd.to_numeric(
    edad_original['Edad'],
    errors='coerce'
)


# Añadimos la edad original al dataframe con el resto de variables
# mediante patient_id
df_edad = df.drop(columns=['Edad'], errors='ignore').merge(
    edad_original,
    on='patient_id',
    how='left',
    validate='one_to_one'
)


print(f'Pacientes en df: {len(df)}')
print(f'Pacientes con edad original disponible: {df_edad["Edad"].notna().sum()}')

display(df_edad[['patient_id', 'Edad']].head())

### Análisis univariante de la edad

La edad es la única variable cuantitativa de la cohorte. Se describe con medidas de tendencia central y dispersión, forma de la distribución y una comprobación orientativa de normalidad. Dado que la normalidad exacta no es necesaria para el EDA, el resultado de Shapiro-Wilk se interpreta junto con los gráficos y los estadísticos robustos (mediana e IQR).

In [ ]:
from scipy.stats import kurtosis, shapiro, skew

# edad original sin normalizar
edad = df_edad['Edad'].dropna()


# Cuartiles e IQR
q1, q3 = edad.quantile([0.25, 0.75])

iqr_edad = q3 - q1


# Test de normalidad
shapiro_stat, shapiro_p = shapiro(edad)


# estad descriptivos
edad_stats = pd.DataFrame({

    'valor': [

        edad.size,
        edad.mean(),
        edad.std(),
        edad.min(),
        q1,
        edad.median(),
        q3,
        iqr_edad,
        edad.max(),
        skew(edad),
        kurtosis(edad, fisher=False),
        shapiro_stat,
        shapiro_p,

    ]

}, index=[

    'n',
    'media',
    'desviación estándar',
    'mínimo',
    'Q1',
    'mediana',
    'Q3',
    'IQR',
    'máximo',
    'asimetría',
    'curtosis de Pearson',
    'Shapiro-Wilk W',
    'Shapiro-Wilk p-valor',

]).round(3)


display(edad_stats)


#graficas
fig, axes = plt.subplots(1, 2, figsize=(14, 4))


# Boxplot
sns.boxplot(
    x=edad,
    color='salmon',
    ax=axes[0]
)

for valor, etiqueta, color in [
    (q1, 'Q1', 'gray'),
    (edad.median(), 'Mediana', 'black'),
    (q3, 'Q3', 'gray')
]:

    axes[0].axvline(
        valor,
        color=color,
        linestyle='--',
        linewidth=1,
        label=etiqueta
    )


axes[0].set_title('Resumen robusto de la edad')
axes[0].set_xlabel('Edad (años)')
axes[0].legend()


# ECDF
sns.ecdfplot(
    x=edad,
    color='firebrick',
    linewidth=2,
    ax=axes[1]
)

axes[1].set_title('Distribución acumulada de la edad')
axes[1].set_xlabel('Edad (años)')
axes[1].set_ylabel('Proporción acumulada')
axes[1].set_ylim(0, 1)


plt.tight_layout()
plt.show()

La cohorte tiene una edad central de 68 años (IQR: 59-76) y un rango de 18 a 87 años. La media (66,0 años) es algo menor que la mediana y la asimetría es negativa, coherente con una cola hacia edades jóvenes. El test de Shapiro-Wilk rechaza normalidad exacta (p < 0,001), por lo que para describir edad conviene informar tanto media/desviación estándar como mediana/IQR.

### Valores extremos según el criterio IQR

Estos valores son observaciones alejadas de la distribución central, no errores automáticamente. Se revisan para confirmar su plausibilidad clínica y de registro, pero no se eliminan por este criterio. Además, en un dataset médico tan pequeño como el nuestro, es imprescindible tener cuantas más muestras mejor, por lo que el estudio de los valores extremos solo es informativo, no eliminativo.

In [ ]:
# limites mediante la regla de 1.5 × IQR
lower = q1 - 1.5 * iqr_edad
upper = q3 + 1.5 * iqr_edad


#pacientes fuera del intervalo
edad_outliers = df_edad[
    (df_edad['Edad'] < lower) |
    (df_edad['Edad'] > upper)
].copy()


print(
    f'Intervalo IQR esperado: '
    f'[{lower:.1f}, {upper:.1f}] años'
)

print(
    f'Observaciones fuera del intervalo: '
    f'{len(edad_outliers)} '
    f'({100 * len(edad_outliers) / len(df_edad):.1f} %)'
)


display(
    edad_outliers[
        ['patient_id', 'Edad', 'Complicacion_binaria']
    ]
)

Todos los outliers son por edades jóvenes. El más extremo es un paciente con 18 años. Lo más normal es que las personas con cáncer de pulmón sean más mayores.

### Distribución por grupos de edad

La agrupación es unicamente informativa porque para el modelado, la edad se conserva como variable continua.

In [ ]:
# creamos grupos de edad a partir de la edad original
grupos_edad = pd.cut(

    edad,

    bins=[0, 40, 50, 60, 70, 80, np.inf],

    labels=[
        '<40',
        '40–49',
        '50–59',
        '60–69',
        '70–79',
        '≥80'
    ],

    right=False,

)


# num de pacientes por grupo
resumen_grupos_edad = (

    grupos_edad
    .value_counts(sort=False)
    .rename_axis('grupo_edad')
    .reset_index(name='n_pacientes')

)


# Porcentaje
resumen_grupos_edad['porcentaje'] = (

    100
    * resumen_grupos_edad['n_pacientes']
    / len(edad)

).round(1)


display(resumen_grupos_edad)


# grafica
fig, ax = plt.subplots(figsize=(8, 4))


sns.barplot(
    data=resumen_grupos_edad,
    x='grupo_edad',
    y='n_pacientes',
    color='salmon',
    ax=ax
)


ax.bar_label(

    ax.containers[0],

    labels=[
        f"{n} ({p:.1f}%)"
        for n, p in zip(
            resumen_grupos_edad['n_pacientes'],
            resumen_grupos_edad['porcentaje']
        )
    ],

    padding=3,

)


ax.set(
    title='Distribución de pacientes por grupo de edad',
    xlabel='Grupo de edad',
    ylabel='Pacientes'
)


plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# DISTRIBUCIÓN DE LAS COMPLICACIONES SEGÚN GRUPO DE EDAD
# ---------------------------------------------------------

# Creamos una columna con los grupos de edad
df_edad['grupo_edad'] = pd.cut(
    df_edad['Edad'],
    bins=[0, 40, 50, 60, 70, 80, np.inf],
    labels=['<40', '40–49', '50–59', '60–69', '70–79', '≥80'],
    right=False
)


target_cols = [
    'Neumotórax',
    'Hemorragia',
    'Sin_complicación'
]


# Pasamos las etiquetas a formato largo
edad_complicaciones = df_edad[
    ['patient_id', 'grupo_edad'] + target_cols
].melt(
    id_vars=['patient_id', 'grupo_edad'],
    value_vars=target_cols,
    var_name='Complicación',
    value_name='Presente'
)


# Número de pacientes de cada grupo de edad
n_por_grupo = (
    df_edad
    .groupby('grupo_edad', observed=False)['patient_id']
    .nunique()
    .rename('n_grupo')
    .reset_index()
)


# Número de casos positivos para cada etiqueta y grupo de edad
resumen_edad_complicaciones = (
    edad_complicaciones
    .groupby(
        ['grupo_edad', 'Complicación'],
        observed=False
    )['Presente']
    .sum()
    .reset_index(name='n_casos')
)


# Añadimos el tamaño total de cada grupo
resumen_edad_complicaciones = resumen_edad_complicaciones.merge(
    n_por_grupo,
    on='grupo_edad',
    how='left'
)


# Calculamos el porcentaje de pacientes del grupo que presentan cada etiqueta
resumen_edad_complicaciones['porcentaje'] = (
    100
    * resumen_edad_complicaciones['n_casos']
    / resumen_edad_complicaciones['n_grupo']
)


# Nombre más legible
resumen_edad_complicaciones['Complicación'] = (
    resumen_edad_complicaciones['Complicación']
    .replace({
        'Sin_complicación': 'Sin complicación'
    })
)


display(resumen_edad_complicaciones.round(1))


# ---------------------------------------------------------
# GRÁFICA MULTIBARRA
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 5))


sns.barplot(
    data=resumen_edad_complicaciones,
    x='grupo_edad',
    y='porcentaje',
    hue='Complicación',
    ax=ax
)


ax.set(
    title='Distribución de complicaciones según grupo de edad',
    xlabel='Grupo de edad',
    ylabel='Pacientes del grupo (%)'
)


ax.legend(
    title='Etiqueta',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)


ax.set_ylim(
    0,
    resumen_edad_complicaciones['porcentaje'].max() * 1.15
)


plt.tight_layout()
plt.show()

Podemos volver a observar como hay una cierta tendencia a edades más mayores. 

In [ ]:
#volvemos a lo original
# Ruta y carga de datos
DATA_PATH = Path('/home/mcribilles/tfm/clinical_data/210pacientes/clinical_data_limpios.csv')
RAW_DATA_PATH = DATA_PATH.with_name('clinical_data.csv')
print(DATA_PATH, '->', DATA_PATH.exists())
print(RAW_DATA_PATH, '->', RAW_DATA_PATH.exists())

df = pd.read_csv(DATA_PATH)
df_raw = pd.read_csv(RAW_DATA_PATH)
print('Shape limpio:', df.shape)
print('Shape original:', df_raw.shape)

Analicemos ahora la variable sexo:

In [ ]:
sex_labels = {0: 'Mujer', 1: 'Hombre'}

sex_counts = (
    df['Sexo_binaria']
    .value_counts()
    .sort_index()
    .rename_axis('Sexo_binaria')
    .reset_index(name='n_pacientes')
)
sex_counts['sexo'] = sex_counts['Sexo_binaria'].map(sex_labels)
sex_counts['porcentaje'] = (100 * sex_counts['n_pacientes'] / len(df)).round(1)
display(sex_counts[['sexo', 'n_pacientes', 'porcentaje']])

sex_comp_counts = pd.crosstab(df['Sexo_binaria'], df['Complicacion_binaria'])
sex_comp = pd.crosstab(df['Sexo_binaria'], df['Complicacion_binaria'], normalize='index')
sex_comp.index = sex_comp.index.map(sex_labels)
sex_comp_counts.index = sex_comp_counts.index.map(sex_labels)
display(sex_comp_counts)
display((sex_comp * 100).round(1))

fig, axes = plt.subplots(1, 2, figsize=(14,5))

sns.countplot(data=df, x='Sexo_binaria', ax=axes[0], color='cornflowerblue')
axes[0].set_title('Distribución de Sexo_binaria')
axes[0].set_xlabel('Sexo')
axes[0].set_ylabel('Pacientes')
axes[0].set_xticklabels(['Mujer', 'Hombre'])
axes[0].bar_label(axes[0].containers[0], padding=3)

sex_comp.plot(kind='bar', stacked=True, ax=axes[1], colormap='Pastel1')
axes[1].set_title('Proporción de complicación binaria por sexo')
axes[1].set_xlabel('Sexo')
axes[1].set_ylabel('Proporción')
axes[1].legend(title='Complicación')
axes[1].set_xticklabels(sex_comp.index, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
sex_labels = {0: 'Mujer', 1: 'Hombre'}

# Etiquetas multietiqueta
label_cols_candidates = [
    'Hemorragia',
    'Neumotórax',
    'Neumotorax',
    'Derrame_pleural',
    'Sin_complicación',
    'Sin_complicacion'
]

label_cols = [col for col in label_cols_candidates if col in df.columns]

if len(label_cols) == 0:
    raise ValueError("No se han encontrado columnas de etiquetas multietiqueta en df.")

# Nombre legible para figuras/tablas
label_names = {
    'Hemorragia': 'Hemorragia',
    'Neumotórax': 'Neumotórax',
    'Neumotorax': 'Neumotórax',
    'Derrame_pleural': 'Derrame pleural',
    'Sin_complicación': 'Sin complicación',
    'Sin_complicacion': 'Sin complicación'
}

# Distribucion global por sexo
sex_counts = (
    df['Sexo_binaria']
    .value_counts()
    .sort_index()
    .rename_axis('Sexo_binaria')
    .reset_index(name='n_pacientes')
)

sex_counts['sexo'] = sex_counts['Sexo_binaria'].map(sex_labels)
sex_counts['porcentaje'] = (100 * sex_counts['n_pacientes'] / len(df)).round(1)

display(sex_counts[['sexo', 'n_pacientes', 'porcentaje']])

# Conteos y porcentajes de cada etiqueta por sexo
rows = []

for sex_value, sex_name in sex_labels.items():
    df_sex = df[df['Sexo_binaria'] == sex_value]
    n_sex = len(df_sex)

    for label in label_cols:
        n_label = int(df_sex[label].fillna(0).astype(int).sum())
        pct_label = 100 * n_label / n_sex if n_sex > 0 else np.nan

        rows.append({
            'sexo': sex_name,
            'etiqueta': label_names.get(label, label),
            'n_pacientes_sexo': n_sex,
            'n_con_etiqueta': n_label,
            'porcentaje_dentro_sexo': round(pct_label, 1)
        })

sex_label_summary = pd.DataFrame(rows)

display(sex_label_summary)

# Matriz de conteos: sexo x etiqueta
sex_label_counts = sex_label_summary.pivot(
    index='sexo',
    columns='etiqueta',
    values='n_con_etiqueta'
)

# Matriz de porcentajes: sexo x etiqueta
sex_label_percentages = sex_label_summary.pivot(
    index='sexo',
    columns='etiqueta',
    values='porcentaje_dentro_sexo'
)

display(sex_label_counts)
display(sex_label_percentages)

# Formato largo para graficas
plot_df = sex_label_summary.copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribucion de sexo
sns.countplot(
    data=df.assign(Sexo=df['Sexo_binaria'].map(sex_labels)),
    x='Sexo',
    ax=axes[0],
    color='cornflowerblue'
)

axes[0].set_title('Distribución de sexo')
axes[0].set_xlabel('Sexo')
axes[0].set_ylabel('Pacientes')

for container in axes[0].containers:
    axes[0].bar_label(container)

# Proporcion de cada etiqueta por sexo
sns.barplot(
    data=plot_df,
    x='etiqueta',
    y='porcentaje_dentro_sexo',
    hue='sexo',
    ax=axes[1]
)

axes[1].set_title('Proporción de etiquetas multietiqueta por sexo')
axes[1].set_xlabel('Etiqueta')
axes[1].set_ylabel('Pacientes con etiqueta dentro de cada sexo (%)')
axes[1].set_ylim(0, max(5, plot_df['porcentaje_dentro_sexo'].max() * 1.15))
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Sexo')

for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f', padding=2)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))

sns.heatmap(
    sex_label_percentages,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    cbar_kws={'label': '% dentro de cada sexo'}
)

plt.title('Proporción de etiquetas multietiqueta por sexo')
plt.xlabel('Etiqueta')
plt.ylabel('Sexo')
plt.tight_layout()
plt.show()


En la cohorte se observa una mayor representación de hombres que de mujeres: 135 hombres frente a 75 mujeres, lo que equivale aproximadamente al 64,3 % y 35,7 % del total, respectivamente. Esta distribución es coherente con el contexto clínico del estudio y con la información aportada por los médicos colaboradores. En generaciones previas, el consumo de tabaco fue más frecuente en hombres. Dado que el tabaquismo es uno de los principales factores de riesgo para el cáncer de pulmón, es esperable que en una cohorte de pacientes candidatos a biopsia pulmonar exista una mayor proporción de varones.

Al analizar la proporción de complicación binaria por sexo, no basta con comparar el número absoluto de complicaciones, ya que hay muchos más hombres que mujeres en el dataset. Por eso se representa la proporción dentro de cada grupo. Con la codificación empleada en el notebook (`0 = mujer`, `1 = hombre`), la tasa de complicación binaria es del 45,3 % en mujeres y del 52,6 % en hombres. Por tanto, en estos datos la complicación aparece ligeramente más frecuente en hombres que en mujeres.

## 6. Prevalencia de variables clínicas y patológicas

En esta parte analizamos qué factores aparecen con mayor frecuencia en la cohorte. Esto es útil tanto para entender el dataset como para priorizar variables en modelos posteriores.

In [ ]:
feature_prev = df[feature_cols].mean().sort_values(ascending=False)
feature_prev_df = pd.DataFrame({
    'variable': feature_prev.index,
    'prevalencia': feature_prev.values,
    'n_positivos': df[feature_prev.index].sum().astype(int).values
})

display(feature_prev_df.head(25))

In [ ]:
# quitamos sin patologia pulmonar y sin factor de riesgo para el grafico
feature_prev_clinical = feature_prev.drop(['Sin_patología_pulmonar', 'Sin_factor_de_riesgo'], errors='ignore')

plt.figure(figsize=(10,8))
feature_prev_clinical.head(20).sort_values().plot(kind='barh', color='cornflowerblue')
plt.title('Top 20 variables clínicas/patológicas más prevalentes')
plt.xlabel('Prevalencia')
plt.show()

### Frecuencia por categoría clínica original

Para distinguir los factores de riesgo de las patologías pulmonares, que en el CSV limpio aparecen mezclados como variables binarias, se representan aquí sus categorías originales. Los valores `X`, vacíos y nulos se excluyen; al ser multietiqueta, un paciente puede aparecer en más de una barra.

In [ ]:
grafico_frecuencias(
    patologias_original, 'Patologías pulmonares más frecuentes', 'steelblue'
)
grafico_frecuencias(
    riesgos_original, 'Factores de riesgo más frecuentes', 'darkseagreen'
)

In [ ]:
# Variables raras: utiles para identificar columnas con soporte muy bajo
rare_vars = feature_prev_df[feature_prev_df['n_positivos'] <= 3].sort_values(['n_positivos', 'variable'])
display(rare_vars)

## 7. Relación exploratoria entre variables y complicación

Este análisis es descriptivo. No implica causalidad ni sustituye la evaluación formal en experimentos posteriores.

### Asociación bivariante mediante test exacto de Fisher y odds ratio

Además de comparar prevalencias, se aplica el test exacto de Fisher para explorar la asociación entre cada variable clínica binaria y `Complicacion_binaria`. El odds ratio permite resumir la dirección y magnitud de la asociación, mientras que la corrección FDR ayuda a controlar el problema de comparaciones múltiples.

In [ ]:
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

rows = []

for col in feature_cols:
    tabla = pd.crosstab(df[col], df['Complicacion_binaria'])

    if tabla.shape == (2, 2):
        odds_ratio, pvalue = fisher_exact(tabla)
        
        tasa_si = df.loc[df[col] == 1, 'Complicacion_binaria'].mean()
        tasa_no = df.loc[df[col] == 0, 'Complicacion_binaria'].mean()
        
        rows.append({
            'variable': col,
            'n_con_variable': int((df[col] == 1).sum()),
            'n_sin_variable': int((df[col] == 0).sum()),
            'tasa_complicacion_si': tasa_si,
            'tasa_complicacion_no': tasa_no,
            'diferencia_tasas': tasa_si - tasa_no,
            'odds_ratio': odds_ratio,
            'pvalue_fisher': pvalue
        })

assoc_df = pd.DataFrame(rows)

assoc_df['pvalue_fdr'] = multipletests(
    assoc_df['pvalue_fisher'],
    method='fdr_bh'
)[1]

assoc_df = assoc_df.sort_values('pvalue_fdr')

display(assoc_df.head(20))

En la ejecución actual no aparece ninguna variable claramente significativa tras corrección por múltiples comparaciones: `Sin_patología_pulmonar` es la señal más cercana al umbral, con una tasa de complicación menor en pacientes sin patología pulmonar registrada y un FDR de 0,068. El resto de variables tienen FDR altos, por lo que deben interpretarse como señales descriptivas. También conviene vigilar los odds ratio extremos o infinitos, porque aparecen en variables con muy pocos pacientes positivos, como EPOC o bronquiectasias, y pueden ser inestables.

### Comparación descriptiva de tasas por complicación binaria

In [ ]:
# Tasa de complicacion binaria en las variables mas prevalentes
selected = feature_prev.head(15).index.tolist()
rate_rows = []
for col in selected:
    positives = df.loc[df[col] == 1, 'Complicacion_binaria']
    rate_rows.append({
        'variable': col,
        'n_con_variable': int((df[col] == 1).sum()),
        'tasa_complicacion': positives.mean() if len(positives) > 0 else np.nan
    })
rate_df = pd.DataFrame(rate_rows).sort_values('tasa_complicacion', ascending=False)
display(rate_df)

plt.figure(figsize=(10,7))
sns.barplot(data=rate_df, y='variable', x='tasa_complicacion', color='tomato')
plt.title('Tasa de complicación binaria en variables prevalentes')
plt.xlim(0, 1)
plt.show()

In [ ]:
# Tasa de etiquetas multietiqueta en las variables mas prevalentes

label_cols_candidates = [
    'Hemorragia',
    'Neumotórax',
    'Neumotorax',
    'Derrame_pleural',
    'Sin_complicación',
    'Sin_complicacion'
]

label_cols = [col for col in label_cols_candidates if col in df.columns]

if len(label_cols) == 0:
    raise ValueError("No se han encontrado columnas de etiquetas multietiqueta en df.")

label_names = {
    'Hemorragia': 'Hemorragia',
    'Neumotórax': 'Neumotórax',
    'Neumotorax': 'Neumotórax',
    'Derrame_pleural': 'Derrame pleural',
    'Sin_complicación': 'Sin complicación',
    'Sin_complicacion': 'Sin complicación'
}

# Variables clinicas mas prevalentes
selected = feature_prev.head(15).index.tolist()

rate_rows = []

for col in selected:
    positives = df[df[col].fillna(0).astype(int) == 1]
    n_positives = len(positives)

    for label in label_cols:
        n_label = int(positives[label].fillna(0).astype(int).sum())
        rate = 100 * n_label / n_positives if n_positives > 0 else np.nan

        rate_rows.append({
            'variable': col,
            'etiqueta': label_names.get(label, label),
            'n_variable': n_positives,
            'n_con_etiqueta': n_label,
            'porcentaje': rate
        })

label_rate_df = pd.DataFrame(rate_rows)

display(
    label_rate_df
    .sort_values(['variable', 'etiqueta'])
    .assign(porcentaje=lambda x: x['porcentaje'].round(1))
)

# Ordenar variables segun la mayor tasa observada en cualquier etiqueta
variable_order = (
    label_rate_df
    .groupby('variable')['porcentaje']
    .max()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(12, 7))

sns.barplot(
    data=label_rate_df,
    y='variable',
    x='porcentaje',
    hue='etiqueta',
    order=variable_order
)

plt.title('Proporción de etiquetas multietiqueta en variables prevalentes')
plt.xlabel('Pacientes con etiqueta dentro de cada variable (%)')
plt.ylabel('Variable')
plt.xlim(0, max(5, label_rate_df['porcentaje'].max() * 1.15))
plt.legend(title='Etiqueta', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
heatmap_df = label_rate_df.pivot(
    index='variable',
    columns='etiqueta',
    values='porcentaje'
)

heatmap_df = heatmap_df.loc[variable_order]

plt.figure(figsize=(9, 7))

sns.heatmap(
    heatmap_df,
    annot=True,
    fmt='.1f',
    cmap='Reds',
    cbar_kws={'label': '% dentro de cada variable'}
)

plt.title('Proporción de etiquetas multietiqueta en variables prevalentes')
plt.xlabel('Etiqueta')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()


In [ ]:
# Comparacion de prevalencia por complicacion binaria
comp0 = df[df['Complicacion_binaria'] == 0][feature_cols].mean()
comp1 = df[df['Complicacion_binaria'] == 1][feature_cols].mean()
prev_compare = pd.DataFrame({
    'sin_complicacion': comp0,
    'con_complicacion': comp1,
    'diferencia': comp1 - comp0
}).sort_values('diferencia', ascending=False)

display(prev_compare.head(20))
display(prev_compare.tail(20))

In [ ]:
plt.figure(figsize=(10,8))
prev_compare['diferencia'].head(15).sort_values().plot(kind='barh', color='darkseagreen')
plt.title('Variables más asociadas exploratoriamente a mayor prevalencia de complicación')
plt.xlabel('Diferencia de prevalencia (con - sin complicación)')
plt.show()

In [ ]:
# Diferencia de prevalencia de variables clinicas segun cada etiqueta multietiqueta

from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

label_cols_candidates = [
    'Hemorragia',
    'Neumotórax',
    'Neumotorax',
    'Derrame_pleural',
    'Sin_complicación',
    'Sin_complicacion'
]

label_cols = [col for col in label_cols_candidates if col in df.columns]

if len(label_cols) == 0:
    raise ValueError("No se han encontrado columnas de etiquetas multietiqueta en df.")

label_names = {
    'Hemorragia': 'Hemorragia',
    'Neumotórax': 'Neumotórax',
    'Neumotorax': 'Neumotórax',
    'Derrame_pleural': 'Derrame pleural',
    'Sin_complicación': 'Sin complicación',
    'Sin_complicacion': 'Sin complicación'
}

# Variables clinicas/patologicas a analizar
feature_cols = feature_prev.index.tolist()

diff_rows = []

for label in label_cols:
    y = df[label].fillna(0).astype(int)

    for col in feature_cols:
        x = df[col].fillna(0).astype(int)

        label_pos = df[y == 1]
        label_neg = df[y == 0]

        n_label_pos = len(label_pos)
        n_label_neg = len(label_neg)

        prev_with_label = x[y == 1].mean() if n_label_pos > 0 else np.nan
        prev_without_label = x[y == 0].mean() if n_label_neg > 0 else np.nan

        diff = prev_with_label - prev_without_label

        # Tabla 2x2 para Fisher:
        # filas: variable presente/ausente
        # columnas: etiqueta presente/ausente
        a = int(((x == 1) & (y == 1)).sum())
        b = int(((x == 1) & (y == 0)).sum())
        c = int(((x == 0) & (y == 1)).sum())
        d = int(((x == 0) & (y == 0)).sum())

        try:
            odds_ratio, p_value = fisher_exact([[a, b], [c, d]])
        except Exception:
            odds_ratio, p_value = np.nan, np.nan

        diff_rows.append({
            'etiqueta': label_names.get(label, label),
            'variable': col,
            'n_con_etiqueta': n_label_pos,
            'n_sin_etiqueta': n_label_neg,
            'prev_con_etiqueta': prev_with_label,
            'prev_sin_etiqueta': prev_without_label,
            'diferencia_prevalencia': diff,
            'odds_ratio': odds_ratio,
            'p_value': p_value
        })

diff_prev_df = pd.DataFrame(diff_rows)

# Correccion por comparaciones multiples dentro de cada etiqueta
diff_prev_df['p_adj_fdr'] = np.nan

for label in diff_prev_df['etiqueta'].unique():
    mask = diff_prev_df['etiqueta'] == label
    pvals = diff_prev_df.loc[mask, 'p_value']

    valid = pvals.notna()
    if valid.sum() > 0:
        _, p_adj, _, _ = multipletests(pvals[valid], method='fdr_bh')
        diff_prev_df.loc[pvals[valid].index, 'p_adj_fdr'] = p_adj

display(
    diff_prev_df
    .sort_values(['etiqueta', 'diferencia_prevalencia'], ascending=[True, False])
    .assign(
        prev_con_etiqueta=lambda x: (100 * x['prev_con_etiqueta']).round(1),
        prev_sin_etiqueta=lambda x: (100 * x['prev_sin_etiqueta']).round(1),
        diferencia_prevalencia=lambda x: (100 * x['diferencia_prevalencia']).round(1),
        p_value=lambda x: x['p_value'].round(4),
        p_adj_fdr=lambda x: x['p_adj_fdr'].round(4)
    )
)

# Seleccionar las variables con mayor diferencia positiva por etiqueta
top_n = 8

plot_df = (
    diff_prev_df
    .dropna(subset=['diferencia_prevalencia'])
    .sort_values(['etiqueta', 'diferencia_prevalencia'], ascending=[True, False])
    .groupby('etiqueta')
    .head(top_n)
    .copy()
)

plot_df['diferencia_prevalencia_pct'] = 100 * plot_df['diferencia_prevalencia']

g = sns.catplot(
    data=plot_df,
    y='variable',
    x='diferencia_prevalencia_pct',
    col='etiqueta',
    kind='bar',
    col_wrap=2,
    height=4,
    aspect=1.4,
    color='mediumseagreen',
    sharex=False,
    sharey=False
)

g.set_axis_labels(
    'Diferencia de prevalencia (con etiqueta - sin etiqueta, puntos porcentuales)',
    'Variable'
)

g.set_titles('{col_name}')

for ax in g.axes.flat:
    ax.axvline(0, color='black', linewidth=0.8)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
main_labels = ['Hemorragia', 'Neumotórax', 'Sin complicación']

plot_df_main = plot_df[plot_df['etiqueta'].isin(main_labels)].copy()

g = sns.catplot(
    data=plot_df_main,
    y='variable',
    x='diferencia_prevalencia_pct',
    col='etiqueta',
    kind='bar',
    col_wrap=3,
    height=4,
    aspect=1.1,
    color='mediumseagreen',
    sharex=False,
    sharey=False
)

g.set_axis_labels(
    'Diferencia de prevalencia, p.p.',
    'Variable'
)

g.set_titles('{col_name}')

for ax in g.axes.flat:
    ax.axvline(0, color='black', linewidth=0.8)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


### Perfiles clínicos de Hemorragia y Neumotórax

Analizamos si `Hemorragia` y `Neumotórax` muestran perfiles clínicos distintos comparando la prevalencia de cada variable clínica en pacientes con y sin cada etiqueta específica.

In [ ]:
targets_especificos = ['Hemorragia', 'Neumotórax']

for target in targets_especificos:
    comp0 = df[df[target] == 0][feature_cols].mean()
    comp1 = df[df[target] == 1][feature_cols].mean()

    tmp = pd.DataFrame({
        'sin_etiqueta': comp0,
        'con_etiqueta': comp1,
        'diferencia': comp1 - comp0,
        'n_con_variable': df[feature_cols].sum()
    }).sort_values('diferencia', ascending=False)

    print(f'\nVariables con mayor diferencia de prevalencia para {target}')
    display(tmp.head(15))

La comparación sugiere que `Hemorragia` y `Neumotórax` no tienen exactamente el mismo perfil clínico. En los pacientes con hemorragia aparecen con mayor prevalencia variables cardiovasculares y respiratorias como hipertensión pulmonar, HTA, enfisema paraseptal, extabaquismo, EPOC y enfisema centrolobulillar. En cambio, en los pacientes con neumotórax destacan más fibrosis, enfisema centrolobulillar, tabaquismo, bronquiectasias y extabaquismo. Esta diferencia es coherente con la idea de que las complicaciones específicas podrían responder a mecanismos clínicos parcialmente distintos. 

## 8. Correlaciones exploratorias

Dado que la mayoría de variables son binarias, este mapa debe interpretarse solo como una señal inicial de asociación lineal simple.

In [ ]:
corr_cols = ['Edad', 'Sexo_binaria', 'Complicacion_binaria', 'Neumotórax', 'Hemorragia', 'Derrame_pleural'] + feature_prev.head(12).index.tolist()
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Mapa de correlaciones (exploratorio)')
plt.show()

In [ ]:
# Correlacion especifica con las etiquetas principales
label_focus = ['Complicacion_binaria', 'Neumotórax', 'Hemorragia', 'Derrame_pleural']
label_corr = df[['Edad', 'Sexo_binaria'] + feature_cols + label_focus].corr(numeric_only=True).loc[feature_cols + ['Edad', 'Sexo_binaria'], label_focus]
display(label_corr.sort_values('Complicacion_binaria', ascending=False).head(20))

## 9. Conclusiones del EDA

El dataset final contiene 210 pacientes y presenta una variable objetivo binaria perfectamente equilibrada: 105 pacientes con complicación y 105 sin complicación. Además, las comprobaciones realizadas muestran que `Complicacion_binaria` es coherente con las complicaciones específicas y que `Sin_complicación` es su complementaria.

En cuanto a las complicaciones específicas (nos vamos a centrar en este proyecto en una clasificación multietiqueta), el soporte es desigual. La etiqueta más frecuente es `Neumotórax`, con 64 casos, seguida de `Hemorragia`, con 49 casos. En cambio, `Derrame_pleural` aparece únicamente en 1 paciente, por lo que no parece viable modelarla de forma independiente con esta muestra. La mayoría de pacientes con complicación tienen una sola complicación etiquetada, aunque existen 9 pacientes con dos complicaciones.

La cohorte está formada mayoritariamente por pacientes de edad avanzada. La mediana de edad es de 68 años, con un rango intercuartílico de 59 a 76 años y un rango total de 18 a 87 años. La distribución no es perfectamente normal y concentra gran parte de los casos en edades medias-altas, algo esperable en una cohorte de pacientes con sospecha o diagnóstico oncológico pulmonar. Respecto al sexo, hay más hombres que mujeres: 135 hombres frente a 75 mujeres. Esta diferencia es clínicamente plausible por el contexto histórico de mayor exposición tabáquica en varones de generaciones previas. La proporción de complicación binaria es ligeramente superior en hombres, 52,6 %, frente a 45,3 % en mujeres, aunque esta comparación es descriptiva y no permite afirmar una asociación independiente sin ajustar por otras variables.

El perfil oncológico está dominado por el adenocarcinoma, con 89 pacientes, seguido del tipo epidermoide, con 38 pacientes, y del grupo no especificado, con 35 pacientes tras agrupar las categorías equivalentes. Otras categorías como linfoma, microcítico, metástasis, neuroendocrino o cáncer de célula no pequeña tienen mucho menor soporte. Las tasas de complicación por tipo de cáncer deben interpretarse con cautela en las categorías pequeñas, ya que pocos pacientes pueden producir porcentajes aparentemente altos o bajos sin que ello implique una diferencia clínicamente sólida.

Las variables clínicas y patológicas más prevalentes son el tabaquismo, la HTA, el extabaquismo y los distintos subtipos de enfisema, especialmente el enfisema centrolobulillar. También se observa que una proporción amplia de pacientes no tiene patología pulmonar registrada. Esta última variable es útil para describir la cohorte, pero conviene interpretarla como ausencia de patología documentada y no como una patología clínica en sí misma. Las patologías pulmonares más frecuentes en el CSV original son el enfisema centrolobulillar, el enfisema paraseptal y la hipertensión pulmonar; entre los factores de riesgo destacan tabaquismo, HTA y extabaquismo.

En el análisis exploratorio de asociación con la complicación binaria no aparecen relaciones fuertes. Algunas variables como fibrosis, EPOC, hipertensión pulmonar, bronquiectasias o enfisema muestran tasas de complicación relativamente elevadas, pero varias tienen muy bajo número de casos, por lo que deben considerarse señales exploratorias y no conclusiones definitivas. Para modelos posteriores sería recomendable priorizar inicialmente variables con suficiente prevalencia y significado clínico claro, controlar las variables extremadamente raras y evitar introducir columnas redundantes o derivadas directamente de la variable objetivo.